<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.2-prompt-optimization/notebooks/exercises-11.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.2 — Prompt Optimization
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

Compaction, output discipline, cache-aware layout, few-shot trade-off, DSPy, A/B in prod.


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.2: Prompt Optimization')


## Ex 1: Token Audit Inventory


In [ ]:
print('Audit a typical 8.7k-token triage bot prompt:')
for component, tokens, fix in [
    ('System preamble ("you are helpful")',  300, 'delete - models infer it'),
    ('Repeated rules (5x same idea)',        1200, 'de-dup to 1 statement'),
    ('5-shot examples',                      2500, 'move 3 to RAG-on-demand'),
    ('Tools schema',                         1800, 'keep, but cache_control'),
    ('Output format prose',                   400, 'replace with JSON schema'),
    ('Repetition + filler',                   500, 'compact'),
    ('Core domain rules',                    2000, 'keep'),
]:
    print(f'  {component:40s} {tokens:>5} tok | fix: {fix}')
print()
print('Total: 8,700 -> ~3,200 raw -> 600 effective with cache_control')


## Ex 2: Output Cost Dominates


In [ ]:
print('Output is 3-5x input -- cutting output saves more:')
in_p, out_p = 3.0, 15.0
for label, in_tok, out_tok in [
    ('baseline',          5000, 800),
    ('-1k input only',    4000, 800),
    ('-500 output only',  5000, 300),
    ('both cuts',         4000, 300),
]:
    cost = (in_tok*in_p + out_tok*out_p) / 1_000_000
    print(f'  {label:20s} in={in_tok}  out={out_tok:>4} -> ${cost:.4f}')
print()
print('-500 out beats -1000 in: same effort, better savings.')


## Ex 3: Cache-Aware Prompt Layout


In [ ]:
print('Order matters for prompt cache hits (Anthropic):')
for slot, content, cacheable in [
    ('1. system rules',       'static role + policies',    'YES (cache_control)'),
    ('2. tools schema',       'JSON schemas of tools',     'YES (cache_control)'),
    ('3. few-shot examples',  'gold input/output pairs',   'YES (cache_control)'),
    ('--- cache boundary ---','',                          ''),
    ('4. RAG chunks',         'per-query context',         'no'),
    ('5. user query',         'this turns input',          'no'),
]:
    print(f'  {slot:24s} {content:32s} | {cacheable}')
print()
print('Mark cache_control on the LAST cacheable block; everything before it caches together.')


## Ex 4: Few-Shot vs Zero-Shot Trade-Off


In [ ]:
print('Few-shot trade-off (illustrative on a triage task):')
for shots, extra_in_tok, quality_pct, with_cache_in_tok in [
    (0,    0, 78,    0),
    (3,  900, 86,   90),
    (5, 1500, 91,  150),
    (8, 2400, 92,  240),
]:
    print(f'  {shots}-shot: +{extra_in_tok:>4} tok | quality {quality_pct}% | with cache: +{with_cache_in_tok} tok')
print()
print('With prompt cache, 5-shot costs ~150 tok extra for +13pp quality. Pick 5.')


## Ex 5: Output Constraints


In [ ]:
print('Output discipline techniques:')
for technique, cuts_tokens_by, notes in [
    ('max_tokens=300',           '40-60%',  'hard cap, may truncate'),
    ('JSON schema (OpenAI/Anth)',  '30-50%','rambling impossible'),
    ('Pydantic + instructor',     '30-50%', 'auto-validation, free retry'),
    ('stop_sequences=["\\n\\n"]', '20-40%','stop early at boundary'),
    ('Anthropic prefill {"',       '5-10%', 'force JSON start'),
    ('Bullet list cap (3 max)',  '30-50%',  'requires prompt instruction'),
]:
    print(f'  {technique:30s}: -{cuts_tokens_by:8s} | {notes}')


## Ex 6: DSPy vs Manual Tuning


In [ ]:
print('Approaches to prompt engineering:')
for approach, gain, when in [
    ('Manual hand-tuning',          '+0-10pp',  'one-off, low volume'),
    ('Few-shot from gold examples', '+5-15pp',  'have labeled data'),
    ('DSPy bootstrap-fewshot',      '+10-20pp', 'stable task, 100+ eval cases'),
    ('DSPy MIPRO',                  '+15-25pp', 'mature task, willing to spend'),
    ('Eval-driven iteration',       '+5-15pp',  'works without DSPy, slower'),
]:
    print(f'  {approach:30s}: {gain:8s} | when: {when}')
print()
print('DSPy bootstrap is a one-time spend; lock the prompt + eval-gate further changes.')


## Ex 7: A/B Test Prompts in Prod


In [ ]:
print('Prompt A/B canary protocol:')
for step, action in [
    ('1. Tag', 'prompt_version on every call'),
    ('2. Canary 5%', 'route 5% to new prompt via feature flag'),
    ('3. Sample eval', '1% of canary judged by LLM judge'),
    ('4. Compare', 'quality / cost / latency / refusal rate vs control'),
    ('5. Auto-rollback', 'on regression > 2pp quality OR > 20% cost'),
    ('6. Promote', 'green for 24h -> 50% -> 100%'),
]:
    print(f'  {step:18s}: {action}')
print()
print('Tools: Langfuse experiments, Braintrust, Promptfoo, in-house feature flag.')


---
## Recap
Audit -> compact -> cache-aware layout -> max_tokens + JSON schema -> few-shot if stable -> DSPy if mature -> A/B in prod.
Output cuts beat input cuts (3-5x cost). Cache_control on stable prefix = 90% read savings.
Real example: 8.7k -> 600 effective tokens, eval +1.2pp, 65% cost cut.
